# 23 · Declared rulesets against an independent decision table

**Question.** Fieldwork reaches a classification by generating N3 facts, binding a threshold
into N3 rules, and handing both to the EYE reasoner. Does that agree with the same logic
written plainly in another language — and is the receipt a reader is shown the same thing the
reasoner was given?

Those are **two claims**, and they fail differently.

1. **The logic produces the same result.** A decision table in Python, derived from each
   ruleset's own prose description, against the conclusions the real reasoner produced. No
   rule engine, no N3, no shared code.
2. **The receipt is the same receipt.** The emitted facts and rules are parsed and checked:
   every asserted predicate is one the ruleset's input contract declares, numbers carry the
   declared precision, and the rules use only the tiny subset this project claims.

**Exhaustive, not sampled.** All three rulesets have finite input spaces — 12, 10 and 24 cells,
enumerated completely. A sample cannot see a cell no rule fires on, a cell two rules disagree
about, or the boundary cell where a value *equals* the threshold. This can.


In [ ]:
# Nothing to load: a decision table is comparisons on values the fixture already carries,
# and the N3 reader is forty lines in the check module. Adding a parser to this kernel was
# declined deliberately — 00-kernel-check records what package combinations kill it.
print("Nothing to load.")


## Parity


In [ ]:
import json, sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_23_rulesets import run

report = run(Path.cwd())
print("environment      :", report["environment"])
print("logic agrees     :", report["logicAgrees"])
print("receipts agree   :", report["receiptsAgree"])
print("cells checked    :", report["totalCells"], "— the complete input space of all three rulesets")
print("agrees           :", report["agrees"])


## The decision tables, every cell

Green is agreement between the reasoner and an independent reading of the description. The
row to look at hardest in each table is the one where the measured value **equals** the
threshold: `math:notGreaterThan` puts it on the safe side, and `math:lessThan` would not.


In [ ]:
from IPython.display import HTML, display

def table(entry):
    rows = entry['rows']
    keys = [k for k in rows[0]['inputs'] if k not in ('subjectId',)]
    head = ''.join(f'<th style="text-align:left;padding:2px 8px">{k}</th>' for k in keys)
    body = ''
    for row in rows:
        tint = '#e6f4ea' if row['agrees'] else '#fce8e6'
        mark = '✓' if row['agrees'] else '✗'
        cells = ''.join(f'<td style="padding:2px 8px">{row["inputs"][k]}</td>' for k in keys)
        emitted = ', '.join(f'{p.split(":")[1]}={v.split(":")[1]}' for p, v in sorted(row['emitted'].items()))
        expected = ', '.join(f'{p.split(":")[1]}={v.split(":")[1]}' for p, v in sorted(row['expected'].items()))
        shown = emitted if row['agrees'] else f'reasoner: {emitted}<br>python: {expected}'
        body += f'<tr style="background:{tint}">{cells}<td style="padding:2px 8px">{shown}</td>'
        body += f'<td style="padding:2px 8px;text-align:center">{mark}</td></tr>'
    return (f'<h4 style="margin:12px 0 4px">{entry["ruleset"].split(":")[-1]} '
            f'<span style="font-weight:normal;color:#666">{entry["rulesetVersion"]} · '
            f'{entry["cells"]} cells · basis {entry["basisStatus"]}</span></h4>'
            f'<table style="border-collapse:collapse;font:12px/1.5 ui-monospace,monospace">'
            f'<tr>{head}<th style="text-align:left;padding:2px 8px">conclusion</th>'
            f'<th style="padding:2px 8px"></th></tr>{body}</table>')

for entry in report['rulesets']:
    display(HTML(table(entry)))


## The receipt, checked rather than trusted

A conclusion a reader cannot audit is worth little. These are the facts and rules the run
actually emitted — not a description of them.

`undeclared` counts predicates asserted into the facts that the ruleset's input contract does
not declare. `instances in rules` is the one that matters most: a ruleset classifies and must
not enumerate, and `nb06-evidence` records a real defect where two facilities are matched by
**name string** — the check shows that the match sits in fact generation, not in the rules.


In [ ]:
for entry in report['rulesets']:
    receipt = entry['receipt']
    print(f"{entry['ruleset'].split(':')[-1]:<20} rules {receipt['ruleCount']:>2}  "
          f"builtins {','.join(b.split(':')[1] for b in receipt['builtins']) or '—':<32} "
          f"undeclared {len(receipt['undeclaredPredicates'])}  "
          f"precision {len(receipt['precisionProblems'])}  "
          f"instances in rules {len(receipt['instancesNamedInRules'])}")
    print(f"{'':<20} declared predicates: {', '.join(p.split(':')[1] for p in receipt['declaredPredicates'])}")
    if receipt['boundValues']:
        print(f"{'':<20} bound into the rules: {', '.join(receipt['boundValues'])}")
    print()


## Dive deeper · the subset is the safety argument

Across all three rulesets the only builtins are `math:greaterThan` and `math:notGreaterThan`.
No negation, no recursion, no aggregation. That is what makes these rules safe to run on a
practitioner's laptop and possible to check exhaustively at all: the input space is finite
because the rules cannot loop.

It is also the argument for entering rules through dropdowns rather than a text box. A text
box admits the rest of N3, and whether a hand-written ruleset terminates is not a check anyone
writes. A builder that can only emit this subset makes the guarantee structural.


In [ ]:
builtins = sorted({b for entry in report['rulesets'] for b in entry['receipt']['builtins']})
print('every builtin used by every shipped ruleset:', ', '.join(builtins))
print('rules in total                            :', sum(e['receipt']['ruleCount'] for e in report['rulesets']))
print('cells, i.e. the complete input space      :', report['totalCells'])
print()
print('a cell no rule fires on would appear here as a missing conclusion;')
print('two rules disagreeing would appear as a red row above.')


## What this cannot establish

That any threshold is the right threshold, or that any of these rulesets is a good one. The
check establishes that the rules say what the ruleset says they say, and that a reader is
shown the same thing the reasoner was given.

And it surfaces something parity cannot fix. One of the three claims a **published** basis,
and none of the three carries evidence a reader can open.


In [ ]:
for entry in report['rulesets']:
    print(f"{entry['ruleset'].split(':')[-1]:<20} basis {entry['basisStatus']:<13} "
          f"resolvable evidence {entry['resolvableEvidence']}  "
          f"named but unresolvable: {', '.join(entry['unresolvedCitations']) or '—'}")
print()
print('rulesets with evidence a reader can open:', report['rulesetsWithResolvableEvidence'], 'of', len(report['rulesets']))


### Reading the result

`logicAgrees` means a decision table written in Python, from each ruleset's prose description,
reproduced every conclusion the EYE reasoner drew across the complete input space.

`receiptsAgree` means the emitted facts assert only predicates the ruleset declares, numbers
carry the declared nine decimals, each cell has one subject, the rules name no dataset
instance, no parameter placeholder was left unbound, and only `math:` builtins appear.

Both are needed. The first without the second is a right answer nobody can audit. The second
without the first is a well-formed receipt for the wrong conclusion.
